In [2]:
import jieba
import re
import csv
from collections import Counter
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# ===================== 文本预处理函数 =====================
def processing(text):
    # 数据清洗
    text = re.sub(r"\{%.+?%\}", " ", text)  # 使用原始字符串避免转义问题
    text = re.sub(r"@.+?( |$)", " ", text)
    text = re.sub(r"【.+?】", " ", text)
    text = re.sub("\u200b", " ", text)
    
    # 分词并过滤非字母/汉字字符
    words = [w for w in jieba.lcut(text) if w.isalpha()]
    
    # 处理否定词
    new_words = []
    skip_next = False
    for i, word in enumerate(words):
        if skip_next:
            skip_next = False
            continue
        if word == "不" and i < len(words) - 1:
            new_words.append(word + words[i+1])
            skip_next = True
        else:
            new_words.append(word)
    
    return " ".join(new_words)

# ===================== 主程序 =====================
if __name__ == "__main__":
    # 读取数据
    csv_path =r"G:\二周\Octopus\情感分析 - 小红书搜索.oec"
    data = []
    
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 2:
                continue
            cleaned = processing(row[1])
            data.append(cleaned)
    
    # 合并文本
    all_text = " ".join(data)
    
    # 词频统计
    words = all_text.split()
    word_counts = Counter(words)
    
    # 自定义停用词
    stopwords = {"正是", "而是", "我们", "那里", "你们", "如今", "说道", "知道", "老太太", 
                "起来", "姑娘", "这里", "出来", "他们", "众人", "自己", "一面", "太太", 
                "只见", "怎么", "奶奶", "两个", "没有", "不是", "不知", "这个", "听见"}
    
    # 过滤停用词并取Top100
    filtered_counts = {word: count for word, count in word_counts.items() if word not in stopwords}
    top_words = dict(Counter(filtered_counts).most_common(100))
    
    # 生成词云
    wc = WordCloud(
        font_path="C:/Windows/Fonts/simhei.ttf",  # 确保路径存在
        background_color="white",
        width=800,
        height=600,
        max_font_size=80,
        collocations=False  # 禁用词组组合
    ).generate_from_frequencies(top_words)
    
    # 显示并保存
    plt.figure(figsize=(10, 8))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title("情感分析词云", fontsize=16)
    plt.tight_layout()
    plt.savefig("情感词云图.jpg", dpi=300, bbox_inches="tight")
    plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'G:\\\\二周\\\\Octopus\\\\情感分析 - 小红书搜索.oec'